### 1. Import Dependencies

In [21]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
dependency_location = os.getenv("DEPENDENCY_LOCATION")
feature_engineered_location = os.getenv("FEATURE_ENGINEERED_LOCATION")
data_processed_location = os.getenv("DATA_PROCESSED_LOCATION")
models_location = os.getenv("MODELS_LOCATION")

In [22]:
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score)

### 2. Read the dataset

In [23]:
df = pd.read_csv(feature_engineered_location)
df.head()

,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,...,InternetService_DSL,InternetService_Fiber optic,InternetService_No,Contract_Month-to-month,Contract_One year,Contract_Two year,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,0,-1.277445,0,0,0,1,0,0,...,1,0,0,1,0,0,0,0,1,0
1,0,0,0,0.066327,1,0,1,0,1,0,...,1,0,0,0,1,0,0,0,0,1
2,0,0,0,-1.236724,1,0,1,1,0,0,...,1,0,0,1,0,0,0,0,0,1
3,0,0,0,0.514251,0,0,1,0,1,1,...,1,0,0,0,1,0,1,0,0,0
4,0,0,0,-1.236724,1,0,0,0,0,0,...,0,1,0,1,0,0,0,0,1,0


In [24]:
df.shape

(7043, 31)

### 3. Model Implementation

#### 3.1 Train-Test Split

In [25]:
RANDOM_STATE = 42

X = df.drop(columns=["Churn"])
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

print(X_train.shape, X_test.shape)
print("Churn rate train:", y_train.mean().round(3), "| test:", y_test.mean().round(3))

(5634, 30) (1409, 30)
Churn rate train: 0.265 | test: 0.265


#### 3.2 Defining Models

In [26]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

baseline_models = {
    "KNN": KNeighborsClassifier(),
    "LogReg": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "RandomForest": RandomForestClassifier(random_state=RANDOM_STATE),
}

#### 3.3 Model Training

In [27]:
scoring = ["roc_auc", "average_precision", "f1", "precision", "recall", "accuracy"]

cv_rows = {}

for model_name, model in baseline_models.items():

    results = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    model_scores = {}

    for metric in scoring:
        mean_score = results[f"test_{metric}"].mean()
        std_score = results[f"test_{metric}"].std()

        model_scores[metric] = f"{mean_score:.3f} ± {std_score:.3f}"

    cv_rows[model_name] = model_scores

cv_df = pd.DataFrame(cv_rows).T

cv_df

,roc_auc,average_precision,f1,precision,recall,accuracy
KNN,0.786 ± 0.010,0.521 ± 0.018,0.549 ± 0.015,0.563 ± 0.010,0.536 ± 0.026,0.767 ± 0.005
LogReg,0.846 ± 0.013,0.661 ± 0.017,0.592 ± 0.030,0.651 ± 0.025,0.544 ± 0.042,0.802 ± 0.012
RandomForest,0.826 ± 0.010,0.622 ± 0.026,0.551 ± 0.026,0.639 ± 0.041,0.484 ± 0.017,0.790 ± 0.015


#### 3.4 Evaluation

In [28]:
def evaluate(model, X_te, y_te, name, setting):
    pred = model.predict(X_te)
    proba = model.predict_proba(X_te)[:, 1]
    return {"Model": name, "Setting": setting,
            "Accuracy": accuracy_score(y_te, pred),
            "Precision": precision_score(y_te, pred),
            "Recall": recall_score(y_te, pred),
            "F1": f1_score(y_te, pred),
            "ROC-AUC": roc_auc_score(y_te, proba),
            "PR-AUC": average_precision_score(y_te, proba)}

In [29]:
rows = []
for name, model in baseline_models.items():
    model.fit(X_train, y_train)
    rows.append(evaluate(model, X_test, y_test, name, "Baseline"))
    joblib.dump(model, os.path.join(models_location, f"{name}_baseline.joblib"))

pd.DataFrame(rows).round(3)

,Model,Setting,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC
0,KNN,Baseline,0.776,0.582,0.551,0.566,0.793,0.533
1,LogReg,Baseline,0.803,0.652,0.551,0.597,0.842,0.635
2,RandomForest,Baseline,0.784,0.614,0.497,0.549,0.823,0.607


### 4. Save Imputed Data Files

In [30]:
split_dir = data_processed_location
X_train.to_csv(os.path.join(split_dir, "X_train.csv"), index=False)
X_test.to_csv(os.path.join(split_dir, "X_test.csv"), index=False)
y_train.to_csv(os.path.join(split_dir, "y_train.csv"), index=False)
y_test.to_csv(os.path.join(split_dir, "y_test.csv"), index=False)